## Basic Installations

In [2]:
# Install LiteEFG
!pip install LiteEFG

# Other utilities
!pip install numpy pandas tqdm

## Create a very basic game file in python

In [3]:
# Write the .game file to disk
# IMPORTANT: num_players is a REQUIRED parameter in the header comment block
game_content = """# Simple 2-depth Coin Matching Game
#
# Two players each have a coin. Player 1 places their coin (Heads or Tails),
# then Player 2 responds. If they match, Player 2 wins; if not, Player 1 wins.
#
# Opt {
#     game_tree: true,
#     num_players: 2,
#     depth: 2,
#     output_file: \"GameInstances/coin_match.txt\",
# }
#
node / player 1 actions H T
node /P1:H player 2 actions H T
node /P1:H/P2:H leaf payoffs 1=-1 2=1
node /P1:H/P2:T leaf payoffs 1=1 2=-1
node /P1:T player 2 actions H T
node /P1:T/P2:H leaf payoffs 1=1 2=-1
node /P1:T/P2:T leaf payoffs 1=-1 2=1
infoset pl1_0__ nodes /
infoset pl2_0__H nodes /P1:H
infoset pl2_1__T nodes /P1:T
"""

game_file_path = "coin_match.game"

with open(game_file_path, "w") as f:
    f.write(game_content)

print(f"Game file written to: {game_file_path}")

Game file written to: coin_match.game


In [4]:
import LiteEFG as efg
import numpy as np
import pandas as pd
from tqdm import trange

## Load the game

In [5]:
# Load the game
env = efg.FileEnv(game_file_path, traverse_type="Enumerate")
print("Game loaded successfully!")

Game loaded successfully!


In [6]:
# Implement CFR directly by inheriting efg.Graph
# Source: official LiteEFG README — there is NO pre-built CFR class to import;
# LiteEFG.baselines.CFR only contains an abstract _baseline stub.

class CFR(efg.Graph):
    def __init__(self):
        super().__init__()

        # --- Static variables (initialised once) ---
        with efg.backward(is_static=True):
            self.strategy      = efg.const(self.action_set_size, 1.0 / self.action_set_size)  # uniform
            self.regret_buffer = efg.const(self.action_set_size, 0.0)                          # zero regrets

        # --- Per-iteration backward pass ---
        with efg.backward():
            # Aggregate counterfactual values from children
            gradient = efg.aggregate(efg.const(1, 0.0), aggregator="sum")
            # Counterfactual regret for each action
            cfr      = gradient - efg.dot(self.strategy, gradient)
            # Accumulate regrets
            self.regret_buffer = self.regret_buffer + cfr
            # Regret-matching: project positive regrets onto the simplex
            self.strategy = efg.normalize(
                efg.maximum(self.regret_buffer, 0.0),
                p_norm=1.0,
                ignore_negative=True
            )

    def update_graph(self, env):
        env.update(self.strategy)

    def current_strategy(self):
        return self.strategy


alg = CFR()
env.set_graph(alg)
print("CFR graph registered.")

CFR graph registered.


In [ ]:
# Run CFR for 1000 iterations, logging exploitability every 100 steps
NUM_ITER   = 1000
PRINT_FREQ = 100
history    = []

for i in trange(NUM_ITER, desc="CFR"):
    alg.update_graph(env)
    env.update_strategy(alg.current_strategy())

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = env.exploitability(alg.current_strategy(), "avg-iterate")
        total = sum(explo)
        history.append({"iteration": i, "exploitability": total})

print("\nDone.")

In [ ]:
# Show exploitability convergence
df_history = pd.DataFrame(history)
print(df_history.to_string(index=False))

In [ ]:
# Run this first to inspect the actual installed source
with open("/opt/anaconda3/lib/python3.11/site-packages/LiteEFG/baselines/CFR.py") as f:
    print(f.read())

In [2]:
# Check ALL available baselines
import os
path = "/opt/anaconda3/lib/python3.11/site-packages/LiteEFG/baselines/"
for f in os.listdir(path):
    print(f)

Reg_CFR.py
CMD.pyi
Balanced_FTRL.py
CMD.py
Balanced_FTRL.pyi
IXOMD.py
__init__.pyi
baseline.py
OS_MCCFR.py
Reg_CFR.pyi
PCFR.pyi
DCFR.pyi
CFRplus.pyi
OS_MCCFR.pyi
CFR.pyi
__init__.py
FTPL.py
CFRplus.py
IXOMD.pyi
__pycache__
DCFR.py
Reg_DOMD.pyi
Balanced_OMD.pyi
utils.py
CFR.py
Balanced_OMD.py
FTPL.pyi
PCFR.py
QFR.py
Reg_DOMD.py
DOMD.pyi
DOMD.py
baseline.pyi
QFR.pyi
MMD.pyi
MMD.py
